In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [3]:
# 1. Load Data
df1 = pd.read_csv('2025_Offense.csv')
df2 = pd.read_csv('2025_offense_epa - Sheet1.csv')

In [4]:
# 2. Clean EPA Data (Convert string percentages to floats)
for col in ['Sack %', 'Scramble %', 'Int %']:
    if df2[col].dtype == 'object':
        df2[col] = df2[col].str.rstrip('%').astype(float) / 100.0

In [11]:
# 3. Merge Data
df = pd.merge(df1, df2, on='Tm', how='inner')
df

,Rk,Tm,G,PF,Cmp,PassAtt,PassYds,PassTD,Int,NY/A,...,Y/A,1stD.1,EPA/Play,Total EPA,EPA/Pass,EPA/Rush,ADoT,Sack %,Scramble %,Int %
0,1,Los Angeles Rams,17,518,388,598,4557,46,8,7.3,...,4.6,126,0.13,136.87,0.23,-0.01,9.51,3.66%,1.11%,1.27%
1,2,New England Patriots,17,490,361,502,4258,31,8,7.7,...,4.4,128,0.13,140.69,0.29,-0.03,9.57,7.82%,10.42%,1.30%
2,3,Seattle Seahawks,17,483,325,481,3877,25,15,7.6,...,4.1,125,0.02,16.57,0.11,-0.07,8.27,5.17%,2.68%,2.87%
3,4,Buffalo Bills,17,481,344,495,3683,29,10,6.9,...,5.0,146,0.12,132.57,0.17,0.08,7.54,6.81%,8.86%,1.70%
4,5,Detroit Lions,17,481,394,582,4303,35,8,6.9,...,4.6,105,0.07,78.79,0.17,-0.06,6.79,6.23%,0.80%,1.28%
5,6,Jacksonville Jaguars,17,474,344,563,3779,29,12,6.3,...,4.0,124,0.03,30.46,0.06,-0.01,8.92,6.34%,6.65%,1.85%
6,7,Dallas Cowboys,17,471,419,624,4527,31,12,6.9,...,4.6,120,0.08,94.83,0.16,-0.02,8.35,4.53%,4.24%,1.75%
7,8,Indianapolis Colts,17,466,360,547,3873,25,14,6.7,...,4.5,122,0.07,69.55,0.06,0.07,8.26,4.81%,4.48%,2.32%
8,9,Chicago Bears,17,441,334,574,3826,28,7,6.4,...,4.9,142,0.07,80.50,0.08,0.06,9.29,3.74%,6.85%,1.09%
9,10,San Francisco 49ers,17,437,398,574,4157,33,16,6.9,...,3.8,119,0.07,77.67,0.16,-0.04,7.81,4.31%,4.15%,2.55%


In [19]:
# 4. Feature Engineering
df["Total Plays"] = df["PassAtt"] + df["RushAtt"]
df["Pass Percent"] = df["PassAtt"] / df["Total Plays"] * 100
df["Rush Percent"] = df["RushAtt"] / df["Total Plays"] * 100
df["Points Per Game"] = df["PF"] / df["G"]
df["Total Yards"] = df["RushYds"] + df["PassYds"]
df["Yards Per Play"] = df["Total Yards"] / df["Total Plays"]
df["Passing Yards Per Attempt"] = df["NY/A"]
df["Rushing Yards Per Attempt"] = df["Y/A"]

In [25]:
# Select all numeric columns for analysis (excluding rank and games played)
df_metrics = df.set_index('Tm').select_dtypes(include=[np.number]).drop(columns=['Rk', 'G'], errors='ignore')

In [26]:
important_stats = [
    'EPA/Rush',
    'EPA/Pass',
    'ADoT',
    'Points Per Game',
    'Yards Per Play',
    'NY/A',
    'PassTD',
    'RushTD',
    'Int',
    'Pass Percent',
    'Rush Percent'
]

plt.figure(figsize=(12, 10))

corr_matrix = df_metrics[important_stats].corr()

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    fmt=".2f",
    annot_kws={"size": 10}
)

plt.title('Correlation Matrix of Key Offensive Metrics', fontsize=18)

plt.tight_layout()
plt.savefig('correlation_matrix_key_metrics.png')
plt.close()

In [22]:
# 6. Scaling Data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_metrics)

In [23]:
# 7. Elbow Method
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42, n_init=10)
    kmeans.fit(scaled_data)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--')
plt.title('Elbow Method For Optimal k (All Columns)')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Within-Cluster Sum of Squares (WCSS)')
plt.xticks(range(1, 11))
plt.grid(True)
plt.tight_layout()
plt.savefig('elbow_method_all.png')
plt.close()

In [28]:
print(df_metrics.columns.tolist())

['PF', 'Cmp', 'PassAtt', 'PassYds', 'PassTD', 'Int', 'NY/A', '1stD', 'RushAtt', 'RushYds', 'RushTD', 'Y/A', '1stD.1', 'EPA/Play', 'Total EPA', 'EPA/Pass', 'EPA/Rush', 'ADoT', 'Total Plays', 'Pass Percent', 'Rush Percent', 'Points Per Game', 'Total Yards', 'Yards Per Play', 'Passing Yards Per Attempt', 'Rushing Yards Per Attempt', 'Cluster']


In [ ]:
# 8. K-Means Clustering (k=6)
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
df_metrics['Cluster'] = kmeans.fit_predict(scaled_data)

# 9. Plot the Clusters based on Passing vs Rushing EPA/Play
plt.figure(figsize=(14, 10))
sns.scatterplot(data=df_metrics, x='EPA/Pass', y='EPA/Rush', hue='Cluster', palette='Set1', s=100)

for team, row in df_metrics.iterrows():
    plt.text(
        row['EPA/Pass'] + 0.003,
        row['EPA/Rush'],
        team,
        fontsize=10
    )
plt.title('K-Means Clustering of NFL Offenses (EPA/Pass vs EPA/Rush)')
plt.tight_layout()
plt.savefig('kmeans_clusters_epa_k4_fixed.png')
plt.close()